# stage10 — Stability, diagram distances, noise, and scientific uncertainty

**Go route** · lab `stage10` · stage

This notebook compiles and runs the **real Go program**. Python is only the
driver: it builds the binary, launches it and compares results. The mathematics
happens in Go.

## The native experiment for this lab

Perturb the same observed image intensity function and measure diagram bottleneck distances in H0/H1.

## Boundary

The same-complex sublevel-set bound does not validate arbitrary preprocessing, correspondence, or semantic significance.

## 1. Setup

In [ ]:
"""Notebook setup. Run this first.

You are in the Go notebook for stage10. It compiles and runs
the real Go program, exactly as native_course.py does, so you can
walk through the code and the evidence interactively.
"""
import importlib.util, json, subprocess, sys, os
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(p for p in [_here, *_here.parents] if (p / "native_course.py").is_file())


def load_runner():
    spec = importlib.util.spec_from_file_location("shape_native_runner", ROOT / "native_course.py")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


runner = load_runner()
LANG = 'go'
LAB_KEY = 'stage10'
LAB = next(r for r in runner.available_labs(ROOT) if r["key"] == LAB_KEY)
RUN_DIR = ROOT / "my_work" / "notebook_stage10_go"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("course root :", ROOT)
print("language    :", LANG)
print("lab         :", LAB["key"], "-", LAB["title"])
print("output dir  :", RUN_DIR.relative_to(ROOT))

## 2. Toolchain check

In [ ]:
# --------------------------------------------------------------------------
# Step 1: is the toolchain present? A missing compiler is an environment
# problem, NOT a mathematics failure. Read native/guides/SETUP.md if it fails.
# --------------------------------------------------------------------------
import shutil

tool = shutil.which('go')
print('go', "->", tool)
if tool:
    version = subprocess.run([tool, *['version']], capture_output=True, text=True)
    print(version.stdout.strip() or version.stderr.strip())
else:
    print("Install the toolchain before continuing: native/guides/SETUP.md")

## 3. Read the dispatch arm for this lab

In [ ]:
# --------------------------------------------------------------------------
# Step 2: read the dispatch arm for THIS lab.
# The dispatch file is a map of lab key -> algorithm. It is not the algorithm.
# Follow the helper calls it makes.
# --------------------------------------------------------------------------
dispatch = ROOT / 'native/go/engine/labs.go'
text = dispatch.read_text(encoding="utf-8")
print("dispatch file:", dispatch.relative_to(ROOT))
print("lines:", len(text.splitlines()))

needle = 'stage10'
for i, line in enumerate(text.splitlines(), start=1):
    if needle in line:
        lo, hi = max(1, i - 3), i + 18
        print(f"--- context around line {i} ---")
        for j in range(lo, hi + 1):
            rows = text.splitlines()
            if j <= len(rows):
                marker = ">>" if j == i else "  "
                print(f"{marker} {j:4d} {rows[j - 1]}")
        break
else:
    print("No direct dispatch arm found for", needle, "- inspect the file yourself.")

## 4. Compile

In [ ]:
# --------------------------------------------------------------------------
# Step 3: compile the native program. This can take a few seconds the first
# time and is cached afterwards under my_work/native_build/.
# --------------------------------------------------------------------------
exe = runner.build(LANG, RUN_DIR, root=ROOT)
print("binary:", exe)
print("sha256:", runner.digest(exe))

## 5. Run this lab

In [ ]:
# --------------------------------------------------------------------------
# Step 4: run ONLY this lab and read the JSON it produced.
# The runner validates the result schema and every consumed-file hash.
# --------------------------------------------------------------------------
result = runner.execute_lab(LANG, LAB_KEY, exe, RUN_DIR, root=ROOT)
print("lab      :", result["lab"])
print("language :", result["language"])
print("status   :", result["status"])
print()
print("metrics:")
print(json.dumps(result["metrics"], indent=2, sort_keys=True))

## 6. Which inputs did it consume?

In [ ]:
# --------------------------------------------------------------------------
# Step 5: what did this program actually read? Every entry is hashed.
# --------------------------------------------------------------------------
for rel, info in sorted(result["inputs"].items()):
    on_disk = runner.digest(ROOT / rel)
    verdict = "MATCH" if on_disk == info["sha256"] else "CHANGED"
    print(rel)
    print("  recorded", info["sha256"])
    print("  on disk ", on_disk, " ", verdict)

## 7. Compare against the other language

Run step 5 in `../rust/stage10.ipynb`, then run
the cell below. You should see agreement within tolerance and identical input
hashes. Agreement is software evidence, not a proof of meaning.

In [ ]:
# --------------------------------------------------------------------------
# Step 6: compare with the other language.
# Open ../rust/stage10.ipynb and run its Step 5 first, then come
# back and run this cell.
# --------------------------------------------------------------------------
OTHER = 'rust'
other = (ROOT / "my_work" / f"notebook_{LAB_KEY}_{OTHER}" / OTHER / f"{LAB_KEY}.json")
print("looking for:", other.relative_to(ROOT))
if not other.is_file():
    print("Not found yet - run Step 4 in the other language notebook first.")
else:
    mine = json.loads((RUN_DIR / LANG / f"{LAB_KEY}.json").read_text(encoding="utf-8"))
    theirs = json.loads(other.read_text(encoding="utf-8"))
    comparison = runner.compare_values(mine["metrics"], theirs["metrics"])
    comparison["input_hashes_identical"] = mine["inputs"] == theirs["inputs"]
    print("agreement within tolerance:", comparison["passed"])
    print("numeric values compared   :", comparison["numeric_values_compared"])
    print("max absolute difference   :", comparison["max_absolute_difference"])
    print("input hashes identical    :", comparison["input_hashes_identical"])
    if comparison["discrepancies"]:
        print("discrepancies:", comparison["discrepancies"])

## 8. Change one thing

`native/guides/LEARNING_PATHS.md`, section D. Go.

Copy the Go source to a private folder, change a single assumption, predict the
effect before running it, then explain what the language made explicit. This
notebook is **unassessed**.

In [ ]:
# --------------------------------------------------------------------------
# Step 7: explain, then change one thing.
# --------------------------------------------------------------------------
# Implementation guide : native/guides/labs/stage10.md
# Language path        : native/guides/LEARNING_PATHS.md  (section D. Go)
# Original Python ref  : notebooks/10_lab.ipynb
#
# Three independent challenges for THIS lab:
# 1. Derive the sup-norm perturbation bound.
# 2. Compare a diagram with the empty diagram, accounting for diagonal matching.
# 3. Explain why unequal essential-class counts lead to infinite extended distance.
#
# Go: is the working matrix a copy of the outer slice, the rows, or both? Change one element of a pivot row and observe whether the caller sees it.
#
# Boundary - what this lab does NOT establish:
# The same-complex sublevel-set bound does not validate arbitrary preprocessing, correspondence, or semantic significance.
#
# Copy native/go to a private folder, change ONE assumption, predict the
# effect BEFORE running, then compare. A copied reference is guided practice.
print("Go notebook complete for", LAB_KEY)